# BTAA Geo API CLI Tutorial

This notebook teaches the `btaa-geo-api` command line tool from the very beginning, then shows how to use it as a composable research tool. You do not need to be a developer. Run each cell in order.

The package installs two command names:

- `btaa-geo-api`: the full, explicit name
- `btaa`: a shorter alias used in some examples

## 1. Install the CLI

In Colab, commands that start with `!` run in a small temporary computer attached to this notebook. The install cell pulls the CLI from the project repository.

In [ ]:
!python -m pip install -q git+https://github.com/geobtaa/api.git#subdirectory=cli

## 2. Check The Help Screen

The help screen lists the available commands. You should see search/discovery commands, resource commands, and newer convenience commands such as `grep`, `context`, `thumbnail`, `static-map`, and `open`.

In [ ]:
!btaa-geo-api --help

## 3. Simple Search

Search for resources about water. The default output is a readable table, which is best for people scanning results in a notebook or terminal.

In [ ]:
!btaa-geo-api --no-analytics search water --per-page 5

## 4. JSON Output

JSON is useful when you want to save, inspect, or process the data later. This returns the API response envelope, including `data` and `meta`.

In [ ]:
!btaa-geo-api --no-analytics search water --per-page 2 --output json

## 5. IDs And Field Extraction

For scripts and notebooks, it is often more useful to print just one value per result. Use `--ids-only` for resource IDs, or `--field` with a dotted field path.

In [ ]:
!btaa-geo-api --no-analytics search water --per-page 5 --ids-only
!btaa-geo-api --no-analytics search water --per-page 5 --field attributes.ogm.dct_title_s

## 6. Save One Resource ID For Later

This cell captures the first search result ID into a Python variable named `RESOURCE_ID`. Later cells use that ID for metadata, citations, thumbnails, static maps, and downloads.

In [ ]:
ids = !btaa-geo-api --no-analytics search water --per-page 1 --ids-only
RESOURCE_ID = ids[0].strip() if ids else ""
print("Using resource:", RESOURCE_ID)

## 7. Discover Fields And Facets

Before filtering, ask the CLI what fields can be used.

In [ ]:
!btaa-geo-api --no-analytics schema facets
!btaa-geo-api --no-analytics schema queryables
!btaa-geo-api --no-analytics schema sortables

## 8. Include Facets

Include filters narrow results to records that match a value.

In [ ]:
!btaa-geo-api --no-analytics search seattle --include gbl_resourceClass_sm=Maps --per-page 5

## 9. Exclude Facets

Exclude filters remove matching records from the result set.

In [ ]:
!btaa-geo-api --no-analytics search seattle --include dct_spatial_sm=Washington --exclude schema_provider_s="Pennsylvania State University" --per-page 5

## 10. Field-Directed Search

This searches only a specific metadata field.

In [ ]:
!btaa-geo-api --no-analytics search Transportation --search-field dct_subject_sm --per-page 5

## 11. Advanced Search

Advanced search passes structured query clauses to the API.

In [ ]:
!btaa-geo-api --no-analytics search "" --adv-q '[{"op":"AND","f":"gbl_resourceClass_sm","q":"Maps"},{"op":"AND","f":"dct_title_s","q":"Island"},{"op":"NOT","f":"dct_title_s","q":"antarctica"}]' --per-page 5

## 12. JSON Lines And Streaming Results

JSON Lines, or `jsonl`, prints one JSON object per line. That makes it easier to process large result sets incrementally.

Use `--all --output jsonl` to fetch every result page and print resources as rows. Use `--stream` as a shortcut for streaming JSON Lines output.

In [ ]:
!btaa-geo-api --no-analytics search railroads --per-page 3 --output jsonl
!btaa-geo-api --no-analytics search "Lake Minnetonka" --per-page 5 --stream

## 13. Search From Standard Input

The CLI can read from standard input. This is what makes it work well in shell pipelines.

Use `-` when stdin contains one query. Use `--each` when stdin contains several newline-delimited queries.

In [ ]:
%%bash
printf "water\n" | btaa-geo-api --no-analytics search - --per-page 3 --ids-only

cat > /tmp/btaa-places.txt <<'EOF'
Minneapolis
Madison
Iowa City
EOF

cat /tmp/btaa-places.txt | btaa-geo-api --no-analytics search --each --per-page 1 --output jsonl

## 14. Grep-Style Discovery

`grep` is a shortcut for fast, pipe-friendly resource discovery. It defaults to JSON Lines, and it has a `--state` shortcut for common spatial filtering.

In [ ]:
!btaa-geo-api --no-analytics grep "soil survey" --state Iowa --per-page 5 --ids-only
!btaa-geo-api --no-analytics grep railroad --per-page 3 --output table

## 15. Metadata, Citations, And Downloads

These commands work with a resource ID. The `RESOURCE_ID` variable came from the earlier search cell.

In [ ]:
print("Using resource:", RESOURCE_ID)
!btaa-geo-api --no-analytics get {RESOURCE_ID} --output json
!btaa-geo-api --no-analytics metadata {RESOURCE_ID} --format ogm
!btaa-geo-api --no-analytics cite {RESOURCE_ID} --format bibtex
!btaa-geo-api --no-analytics downloads {RESOURCE_ID}

## 16. Batch Download Pipelines

You can connect search output directly to downloads. This example keeps the result set tiny and uses `--no-clobber` so rerunning the notebook does not overwrite files.

Some resources may not have downloadable files. If this cell reports that multiple downloads are available or no download is available, try changing the search query or adding `--type`.

In [ ]:
%%bash
mkdir -p /content/btaa-downloads
btaa-geo-api --no-analytics search "geojson" --per-page 2 --ids-only \
  | btaa-geo-api --no-analytics download --ids - --best --out /content/btaa-downloads --no-clobber || true

find /content/btaa-downloads -maxdepth 1 -type f -print

## 17. Thumbnails And Static Maps

The CLI can fetch preview images for a resource. These are useful for teaching materials, QA, slides, and quick visual inspection.

In [ ]:
from pathlib import Path
from IPython.display import Image, SVG, display

thumb_path = "/content/btaa-thumbnail.img"
map_path = "/content/btaa-static-map.img"

!btaa-geo-api --no-analytics thumbnail {RESOURCE_ID} --out {thumb_path}
!btaa-geo-api --no-analytics static-map {RESOURCE_ID} --out {map_path}

def show_image(path):
    data = Path(path).read_bytes()
    if data.lstrip().startswith(b"<svg"):
        display(SVG(filename=path))
    else:
        display(Image(filename=path))

print("Thumbnail:")
show_image(thumb_path)
print("Static map:")
show_image(map_path)

## 18. LLM-Ready Research Context

The `context` command creates compact Markdown or JSON for notebooks, AI tools, and quick research scans. It is not a replacement for reading the records; it is a convenient starting packet.

In [ ]:
!btaa-geo-api --no-analytics context "Mississippi River maps" --per-page 5

## 19. Open Resource URLs

In Colab, `open` prints a URL you can click. In a local terminal, add `--browser` to launch the system browser.

In [ ]:
!btaa-geo-api --no-analytics open {RESOURCE_ID}
!btaa-geo-api --no-analytics open "dakota county parcels"

## 20. Shell Completion

On your own computer, Typer can install shell completion for bash, zsh, fish, and PowerShell. Completion is less useful inside Colab, but these are the commands to run locally.

In [ ]:
!btaa-geo-api --show-completion bash | head -n 20

## 21. Aardvark Validation And Metadata Crosswalks

The CLI can validate an OGM Aardvark JSON record and can crosswalk ISO 19139 or FGDC XML into Aardvark JSON. The crosswalks follow the same core field families used by GeoCombine: title, description, creators, publishers, provider, rights, resource class/type, subjects, places, dates, bounding boxes, and references.

In [ ]:
import json
from pathlib import Path

sample_record = {
    "id": "demo-record",
    "dct_title_s": "Demo Aardvark Record",
    "gbl_resourceClass_sm": ["Datasets"],
    "dct_accessRights_s": "Public",
    "gbl_mdVersion_s": "Aardvark",
}
Path("/content/demo-aardvark.json").write_text(json.dumps(sample_record), encoding="utf-8")
!btaa-geo-api --no-analytics validate /content/demo-aardvark.json --output table

In [ ]:
%%bash
cat > /content/demo-fgdc.xml <<'EOF'
<metadata>
  <idinfo>
    <citation><citeinfo><origin>Demo Lab</origin><pubdate>20200115</pubdate><title>Demo roads</title><geoform>vector digital data</geoform><onlink>https://example.test/roads</onlink></citeinfo></citation>
    <descript><abstract>Demo road centerlines.</abstract></descript>
    <keywords><theme><themekey>transportation</themekey></theme><place><placekey>Iowa</placekey></place></keywords>
    <spdom><bounding><westbc>-94</westbc><eastbc>-90</eastbc><northbc>44</northbc><southbc>40</southbc></bounding></spdom>
    <accconst>None</accconst>
  </idinfo>
</metadata>
EOF

btaa-geo-api --no-analytics crosswalk /content/demo-fgdc.xml --from fgdc --validate
btaa-geo-api --no-analytics crosswalks --from fgdc --output table

## 22. Map The News

A good geospatial search tool should help you move from a headline to context. When a story mentions flooding, transit, housing, elections, wildfire smoke, agriculture, or public health, the next useful question is often spatial: **what maps, datasets, aerial imagery, and historical records help explain where this is happening?**

This chapter is a small, blog-style workflow for doing that with the CLI. We will start with a newsy topic, turn it into place-aware searches, collect matching BTAA resources, and draw an interactive map from the bounding boxes in the metadata.

### The Story Seed

For a proof of concept, imagine we are preparing background material for a story about Mississippi River flooding and infrastructure. We will search across a handful of river states. You can swap these strings for any story you care about.

In [ ]:
NEWS_TOPIC = "Mississippi River flooding infrastructure"
NEWS_PLACES = ["Minnesota", "Wisconsin", "Iowa", "Illinois", "Missouri"]

print("Topic:", NEWS_TOPIC)
print("Places:", ", ".join(NEWS_PLACES))

### First Pass: Search Like A Reporter

Before building a map, do a quick terminal-style pass. The CLI can read newline-delimited searches from standard input, so we can generate one query per place and stream compact JSON Lines back.

In [ ]:
%%bash
cat > /tmp/map-the-news-queries.txt <<'EOF'
Mississippi River flooding infrastructure Minnesota
Mississippi River flooding infrastructure Wisconsin
Mississippi River flooding infrastructure Iowa
Mississippi River flooding infrastructure Illinois
Mississippi River flooding infrastructure Missouri
EOF

cat /tmp/map-the-news-queries.txt   | btaa-geo-api --no-analytics search --each --per-page 2 --output jsonl   | head -n 10

### Full Proof Of Concept: Search, Extract Footprints, Draw A Map

The next cell is intentionally complete. It calls the CLI from Python, parses the JSON response, extracts `ENVELOPE(west, east, north, south)` bounding boxes, and renders an interactive Folium map. This is the basic pattern you could reuse for a newsroom notebook, course assignment, exhibit planning session, or research briefing.

The map is not claiming that every resource is directly about the news event. It is a **context map**: a way to quickly find geospatial material near the places and themes in the story.

In [ ]:
!python -m pip install -q folium

import html
import json
import re
import subprocess
from pathlib import Path

import folium
from IPython.display import HTML, display

ENVELOPE_RE = re.compile(
    r"ENVELOPE\(\s*([-0-9.]+)\s*,\s*([-0-9.]+)\s*,\s*([-0-9.]+)\s*,\s*([-0-9.]+)\s*\)"
)


def run_cli_search(query, per_page=5):
    command = [
        "btaa-geo-api",
        "--no-analytics",
        "search",
        query,
        "--per-page",
        str(per_page),
        "--output",
        "json",
    ]
    completed = subprocess.run(command, capture_output=True, text=True, check=False)
    if completed.returncode != 0:
        print("Search failed:", query)
        print(completed.stderr or completed.stdout)
        return []
    payload = json.loads(completed.stdout)
    return payload.get("data", [])


def ogm_attributes(item):
    attrs = item.get("attributes", {}) if isinstance(item, dict) else {}
    return attrs.get("ogm", attrs) if isinstance(attrs, dict) else {}


def first_value(value):
    if isinstance(value, list):
        return value[0] if value else ""
    return value or ""


def parse_envelope(value):
    if isinstance(value, list):
        value = " ".join(str(part) for part in value)
    match = ENVELOPE_RE.search(str(value or ""))
    if not match:
        return None
    west, east, north, south = map(float, match.groups())
    if west == east or north == south:
        return None
    return west, east, north, south


resources_by_id = {}
for place in NEWS_PLACES:
    query = f"{NEWS_TOPIC} {place}"
    for item in run_cli_search(query, per_page=5):
        resource_id = item.get("id")
        if not resource_id:
            continue
        item["_news_place"] = place
        item["_news_query"] = query
        resources_by_id.setdefault(resource_id, item)

print(f"Collected {len(resources_by_id)} unique resources")

m = folium.Map(location=[42.8, -91.5], zoom_start=5, tiles="CartoDB positron")
rows_without_bbox = []
plotted = 0

for item in resources_by_id.values():
    ogm = ogm_attributes(item)
    bbox = parse_envelope(ogm.get("dcat_bbox") or ogm.get("locn_geometry"))
    title = first_value(ogm.get("dct_title_s")) or item.get("id", "Untitled")
    provider = first_value(ogm.get("schema_provider_s"))
    year = first_value(ogm.get("gbl_indexYear_im"))
    resource_id = item.get("id", "")
    if not bbox:
        rows_without_bbox.append((resource_id, title))
        continue
    west, east, north, south = bbox
    popup_html = f"""
    <strong>{html.escape(str(title))}</strong><br>
    <code>{html.escape(str(resource_id))}</code><br>
    {html.escape(str(provider))}<br>
    {html.escape(str(year))}<br>
    Query: {html.escape(str(item.get('_news_query', '')))}
    """
    folium.Rectangle(
        bounds=[[south, west], [north, east]],
        color="#2563eb",
        weight=2,
        fill=True,
        fill_opacity=0.12,
        popup=folium.Popup(popup_html, max_width=420),
        tooltip=title,
    ).add_to(m)
    plotted += 1

map_path = Path("/content/map-the-news.html")
m.save(map_path)

print(f"Mapped {plotted} resources with bounding boxes")
if rows_without_bbox:
    print(f"Skipped {len(rows_without_bbox)} resources without parseable bounding boxes")

display(HTML(f'<iframe src="{map_path}" width="100%" height="620"></iframe>'))

### What To Do With The Map

Use the popups as leads. A useful next pass might download candidate datasets, fetch thumbnails or static maps for promising records, or generate an LLM-ready context packet for the same topic.

In [ ]:
!btaa-geo-api --no-analytics context "Mississippi River flooding infrastructure" --per-page 5

### Adapt The Pattern

Try changing `NEWS_TOPIC` and `NEWS_PLACES` above. A few ideas:

- `urban heat tree canopy` with `Chicago`, `Detroit`, `Minneapolis`, `Madison`
- `railroad expansion history` with `Iowa`, `Indiana`, `Ohio`, `Illinois`
- `wildfire smoke air quality` with `Minnesota`, `Wisconsin`, `Michigan`
- `election precinct boundaries` with a set of states or counties

The key move is the same each time: turn a story into places, places into metadata searches, and metadata footprints into a map you can inspect.

## Troubleshooting

- If install fails, restart the runtime and run the install cell again.
- If you see a rate-limit message, wait a minute or configure an API key.
- To use a different server, set `BTAA_GEO_API_BASE_URL` before running commands.
- If an image or download cell fails for a specific resource, run another search and set `RESOURCE_ID` to a different ID.
- For large searches, prefer `--output jsonl`, `--stream`, `--ids-only`, or `--field` so the notebook stays responsive.